# P2.4 — GTAT structural/temporal expert — confirmation seed 8675309

Retrains the frozen P2.4 expert with training seed **8675309** for both target folds (BGL, HDFS). Only the training seed changes: data, config, step budget and checkpoint-selection rule are those of seed 42, and for P2.2–P2.4 the selection windows are also those of seed 42.

**Seed 42 stays the primary seed.** This notebook never writes to the seed-42 folders and changes nothing in P2-FREEZE-001 or Phase 3. Protocol: `docs/protocol/SEED-CONFIRM-001-seed-confirmation.md` (also inside the code ZIP).

## Setup

1. In Colab, choose **Runtime → Change runtime type** and select any GPU with at least 12 GiB (T4, L4, A100, H100 or G4). P2.4 trains in FP32, so BF16 is not required.
2. Upload `seqlogad-code-seeds-v1.zip` once to `MyDrive/SeqLogAD/Seeds/uploads/` (both seeds share it). The data ZIPs are read from the seed-42 upload folders (`MyDrive/SeqLogAD/P2.4_GTAT/uploads/`, `MyDrive/SeqLogAD/P2.2_Sequence/uploads/`); no second upload is needed.
3. No Hugging Face token is needed.
4. Choose **Runtime → Run all**.

Outputs go to `MyDrive/SeqLogAD/Seeds/S8675309/P2.4/`. If Colab disconnects, run the notebook again: finished parts are skipped and an unfinished fold resumes from its last recovery checkpoint. Disconnect only after `SAFE_TO_DISCONNECT_COLAB=YES`.


In [ ]:
# HUMAN CONFIG — seed/fold are fixed by the development protocol.
from pathlib import Path
import os, uuid
from google.colab import drive

drive.mount("/content/drive")

TASK_ID = "P2.4"
NOTEBOOK_STATUS = "READY"
BLOCKERS = []
RUN_MODE = "development"
SEED = 8675309  # SEED-CONFIRM-001 confirmation seed; seed 42 stays primary
TARGETS = ["BGL", "HDFS"]  # EXC-007: Hadoop/OpenStack are source-only
DRIVE_ROOT = Path("/content/drive/MyDrive/SeqLogAD/Seeds/S8675309")  # this seed's own folder
UPLOAD_ROOT = Path("/content/drive/MyDrive/SeqLogAD/Seeds/uploads")  # the seed code ZIP, shared by both seeds
# Data ZIPs come from the seed-42 upload folders; nothing is written there.
UPLOAD_ROOTS = [UPLOAD_ROOT] + [Path("/content/drive/MyDrive/SeqLogAD") / folder / "uploads" for folder in ("P2.4_GTAT", "P2.2_Sequence",)]
PERSISTENT_ROOT = DRIVE_ROOT
# Keep Python, extracted source and temporary outputs on Colab's local disk.
STATE_ROOT = Path("/content/SeqLogAD")
HF_HOME = STATE_ROOT / "hf-cache"
VENV = STATE_ROOT / "venv-py312"
RUNTIME_ROOT = STATE_ROOT / "runtime" / "seeds-ecb9a81deb2d-P2.4"
REPO_ROOT = str(RUNTIME_ROOT / "seqlogad_code_seed_confirmation")
DATA_ROOT = str(RUNTIME_ROOT / "phase2_sequence")
SOURCE_DEVELOPMENT_ROOT = str(RUNTIME_ROOT / "phase2_source_development_v13r1")
GRAPH_ROOT = str(RUNTIME_ROOT / "phase2_graph")
SESSION_ROOT = Path("/content/seqlogad_sessions") / uuid.uuid4().hex
OUTPUT_ROOT = str(SESSION_ROOT / "seqlogad_outputs")
EXPECTED_CODE_ZIP_SHA256 = "ecb9a81deb2df85e9eb2abfffd571c75ec7340de7d059cf145a4d91029d44506"
EXPECTED_DATA_ZIP_SHA256 = "4ed76b92e4b001138b28b4b3b4d3017d6ff7f30dd31e8cca0649bfdf34ede781"
EXPECTED_SOURCE_DEVELOPMENT_ZIP_SHA256 = "c765eb88a4feebc8ceb157110e5a6bd28dec942a7fb9343d7f6b969b3e79b295"
EXPECTED_BUNDLE_SHA256 = "1268b669a8b1b474658cfa7daa6556c987b0f31f07e0101ed30af250a8d7f129"
EXPECTED_SOURCE_DEVELOPMENT_MANIFEST_SHA256 = "885b53821671e7f60e6df5270a78a23535900bf4ba6d339e69b29f93eb2c6399"
EXPECTED_GRAPH_ZIP_SHA256 = "595b5dc7ec1b96abc80b4722b5f39dcf1c5653ec25aefebcce7ee74dbf36f9ac"
EXPECTED_GRAPH_MANIFEST_SHA256 = "8918172b4376fd822ec812e9b46b2725f83b517a43c94f3efe16f55623c9d059"
MIN_VRAM_GIB = 12

assert RUN_MODE == "development" and SEED in (3407, 8675309)
assert TARGETS == ["BGL", "HDFS"]
assert NOTEBOOK_STATUS == "READY", "Notebook is fail-closed: " + ", ".join(BLOCKERS)
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
for directory in (STATE_ROOT, PERSISTENT_ROOT, HF_HOME, SESSION_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
print({"session_root": str(SESSION_ROOT), "persistent_root": str(PERSISTENT_ROOT),
       "upload_folders": [str(root) for root in UPLOAD_ROOTS]})


In [ ]:
# Colab normally includes uv; install it only if absent.
import sys, shutil, subprocess

uv_real = shutil.which("uv")
if not uv_real:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.8.17"], check=True)
    uv_real = shutil.which("uv")
assert uv_real, "uv installation failed"
uv_expected = Path(sys.executable).parent / "uv"
if not uv_expected.exists():
    uv_expected.symlink_to(uv_real)
print("uv:", uv_real)


## Verify uploads, extract atomically and build the frozen runtime


In [ ]:
from pathlib import PurePosixPath
import hashlib, json, shutil, stat, subprocess, sys, zipfile

def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

def locate(expected):
    # First match wins, so a ZIP copied into both upload folders is not ambiguous.
    for root in UPLOAD_ROOTS:
        for path in (sorted(root.glob("*.zip")) if root.is_dir() else []):
            if digest(path) == expected:
                return path
    raise AssertionError(f"Put the ZIP with SHA-256 {expected} in one of {[str(r) for r in UPLOAD_ROOTS]}")

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as handle:
        infos = handle.infolist()
        names = [item.filename for item in infos]
        assert len(names) == len(set(names)), "Duplicate ZIP member"
        for item in infos:
            member = PurePosixPath(item.filename)
            assert not member.is_absolute() and ".." not in member.parts and "\\" not in item.filename
            assert ((item.external_attr >> 16) & 0o170000) != stat.S_IFLNK, "ZIP symlink forbidden"
        assert handle.testzip() is None, "ZIP CRC failure"
        handle.extractall(destination)

code_zip = locate(EXPECTED_CODE_ZIP_SHA256)
data_zip = locate(EXPECTED_DATA_ZIP_SHA256)
label_zip = locate(EXPECTED_SOURCE_DEVELOPMENT_ZIP_SHA256)
graph_zip = locate(EXPECTED_GRAPH_ZIP_SHA256)
marker = RUNTIME_ROOT / "EXTRACTION.json"
expected_marker = {
    "code_sha256": EXPECTED_CODE_ZIP_SHA256,
    "data_sha256": EXPECTED_DATA_ZIP_SHA256,
    "source_dev_sha256": EXPECTED_SOURCE_DEVELOPMENT_ZIP_SHA256,
    "graph_sha256": EXPECTED_GRAPH_ZIP_SHA256,
}
if RUNTIME_ROOT.exists():
    assert marker.is_file(), f"Unverified runtime directory exists: {RUNTIME_ROOT}"
    assert json.loads(marker.read_text()) == expected_marker, "Runtime extraction identity mismatch"
else:
    partial = RUNTIME_ROOT.with_name(RUNTIME_ROOT.name + ".partial")
    if partial.exists():
        shutil.rmtree(partial)
    partial.mkdir(parents=True)
    safe_extract(code_zip, partial)
    safe_extract(data_zip, partial)
    safe_extract(label_zip, partial)
    safe_extract(graph_zip, partial)
    assert (partial / "seqlogad_code_seed_confirmation").is_dir()
    assert (partial / "phase2_sequence/manifests/bundle.json").is_file()
    assert (partial / "phase2_source_development_v13r1/manifest.json").is_file()
    assert (partial / "phase2_graph/manifests/graph-bundle.json").is_file()
    (partial / "EXTRACTION.json").write_text(json.dumps(expected_marker, sort_keys=True) + "\n")
    partial.rename(RUNTIME_ROOT)

assert Path(REPO_ROOT).is_dir()
assert Path(DATA_ROOT, "manifests/bundle.json").is_file()
assert digest(Path(DATA_ROOT) / "manifests/bundle.json") == EXPECTED_BUNDLE_SHA256
assert digest(Path(SOURCE_DEVELOPMENT_ROOT) / "manifest.json") == EXPECTED_SOURCE_DEVELOPMENT_MANIFEST_SHA256
assert digest(Path(GRAPH_ROOT) / "manifests/graph-bundle.json") == EXPECTED_GRAPH_MANIFEST_SHA256
print({"code_zip": str(code_zip), "data_zip": str(data_zip), "label_zip": str(label_zip), "graph_zip": str(graph_zip),
       "extraction": "PASS"})

def isolated_env():
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    return env

def run_live(args, check=True):
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=isolated_env()) as process:
        for line in process.stdout:
            token = os.environ.get("HF_TOKEN")
            if token:
                line = line.replace(token, "[TOKEN_HIDDEN]")
            print(line, end="", flush=True)
        code = process.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, args)
    return code

run_live(["nvidia-smi"])
PYTHON = str(VENV / "bin/python")
if not Path(PYTHON).is_file():
    if sys.version_info[:2] == (3, 12):
        run_live([sys.executable, "-m", "pip", "install", "virtualenv==20.31.2"])
        run_live([sys.executable, "-m", "virtualenv", "--python", sys.executable, str(VENV)])
    else:
        run_live([sys.executable, "-m", "pip", "install", "uv==0.8.17"])
        uv = str(Path(sys.executable).parent / "uv")
        assert Path(uv).is_file(), "uv bootstrap executable missing"
        run_live([uv, "python", "install", "3.12"])
        run_live([uv, "venv", "--python", "3.12", "--seed", str(VENV)])

run_live([PYTHON, "-c", "import sys; assert sys.version_info[:2] == (3, 12); print(sys.version)"])
run_live([PYTHON, "-m", "pip", "install", "torch==2.9.1", "--index-url",
          "https://download.pytorch.org/whl/cu128"])
run_live([PYTHON, "-m", "pip", "install", "-e", REPO_ROOT, "packaging>=24,<27"])
run_live([PYTHON, "-c", "from seqlogad.semantic.runtime import install_missing; install_missing(" + repr(REPO_ROOT) + ")"])
hardware_check = f"""
import json, torch
assert torch.cuda.is_available(), "CUDA is unavailable"
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info(0)
native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
capability = torch.cuda.get_device_capability(0)
assert total >= {MIN_VRAM_GIB} * 1024**3, f"Expected at least {MIN_VRAM_GIB} GiB VRAM; got {{total / 1024**3:.1f}} GiB on {{name}}"
print(json.dumps({{"gpu": name, "compute_capability": capability, "free_vram_bytes": free, "total_vram_bytes": total,
                  "native_bf16": native_bf16, "torch": torch.__version__,
                  "cuda": torch.version.cuda}}, indent=2))
"""
run_live([PYTHON, "-c", hardware_check])
run_live([PYTHON, "-m", "seqlogad.sequence.cli", "environment", "--repo-root", REPO_ROOT])


## Sequential target loop with persistent recovery

In [ ]:
# Trains BGL then HDFS in one process; each optimizer step prints one JSON metrics line.
args = [
    PYTHON, "-m", "seqlogad.seeds.runner",
    "--task-id", TASK_ID,
    "--seed", str(SEED),
    "--repo-root", REPO_ROOT,
    "--data-root", DATA_ROOT,
    "--output-root", OUTPUT_ROOT,
    "--drive-root", str(PERSISTENT_ROOT),
    "--expected-bundle-sha256", EXPECTED_BUNDLE_SHA256,
    "--source-development-root", SOURCE_DEVELOPMENT_ROOT,
    "--expected-source-development-sha256", EXPECTED_SOURCE_DEVELOPMENT_MANIFEST_SHA256,
    "--graph-root", GRAPH_ROOT,
    "--expected-graph-sha256", EXPECTED_GRAPH_MANIFEST_SHA256,
]
run_live(args, check=True)


## Final persistent-integrity and safe-stop gate


In [ ]:
final_zip = PERSISTENT_ROOT / TASK_ID / f"seqlogad_outputs_P2.4_S{SEED}_FINAL.zip"
final_sidecar = Path(str(final_zip) + ".sha256")
assert final_zip.is_file() and final_sidecar.is_file(), "Verified final persistent ZIP is missing"
declared_hash, declared_name = final_sidecar.read_text().strip().split("  ", 1)
assert declared_name == final_zip.name and digest(final_zip) == declared_hash
# Output must hold one trained GTAT model archive per target.
with zipfile.ZipFile(final_zip) as handle:
    members = set(handle.namelist())
    summary = json.loads(handle.read(f"P2.4_S{SEED}/SUMMARY.json"))
expected_targets = {f"P2.4_S{SEED}/targets/P2.4_{t}_S{SEED}.zip" for t in TARGETS}
assert expected_targets <= members, f"Missing P2.4 target archives: {sorted(expected_targets - members)}"
assert sorted(summary["targets"]) == sorted(TARGETS), summary["targets"]
assert summary["source_label_status"] == "SOURCE_VALIDATION_MODEL_SELECTION_ONLY"
shutil.rmtree(SESSION_ROOT)
print("====================================")
print("SEQLOGAD COLAB EXECUTION SUMMARY — P2.4 GTAT STRUCTURAL/TEMPORAL")
print(f"GTAT_TARGETS={TARGETS}")
print(f"FINAL_ZIP={final_zip}")
print(f"FINAL_ZIP_SIZE={final_zip.stat().st_size}")
print(f"FINAL_ZIP_SHA256={declared_hash}")
print("PERSISTENT_STORAGE_VERIFIED=YES")
print("SAFE_TO_DISCONNECT_COLAB=YES")
print("====================================")